In [19]:
import os

for folder in ['bronze', 'silver', 'gold']:
    os.makedirs(folder, exist_ok=True)

In [20]:
import urllib.request
import zipfile

url = "https://cdn.freecodecamp.org/project-data/books/book-crossings.zip"
zip_path = "book-crossings.zip"

# Configure a User-Agent header to prevent 403 Forbidden errors
req = urllib.request.Request(
    url,
    headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'}
)

with urllib.request.urlopen(req) as response, open(zip_path, 'wb') as out_file:
    out_file.write(response.read())

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("raw_data")

In [21]:
import pandas as pd

users_raw = pd.read_csv("raw_data/BX-Users.csv", sep=";", encoding="latin-1", on_bad_lines="skip", low_memory=False)

In [22]:
from datetime import datetime, timezone
import pandas as pd

# Ensure all raw datasets are loaded
if 'books_raw' not in globals():
    books_raw = pd.read_csv("raw_data/BX-Books.csv", sep=";", encoding="latin-1", on_bad_lines="skip", low_memory=False)
if 'ratings_raw' not in globals():
    ratings_raw = pd.read_csv("raw_data/BX-Book-Ratings.csv", sep=";", encoding="latin-1", on_bad_lines="skip", low_memory=False)

ingest_time = datetime.now(timezone.utc).isoformat()

for df, name in [(users_raw, "users"), (books_raw, "books"), (ratings_raw, "ratings")]:
    df["ingestion_timestamp"] = ingest_time
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str)
    df.to_parquet(f"bronze/{name}.parquet", index=False)

In [23]:
# Load the raw data to inspect true missing values and data types
users_eda = pd.read_csv("raw_data/BX-Users.csv", sep=";", encoding="latin-1", on_bad_lines="skip", low_memory=False)
books_eda = pd.read_csv("raw_data/BX-Books.csv", sep=";", encoding="latin-1", on_bad_lines="skip", low_memory=False)
ratings_eda = pd.read_csv("raw_data/BX-Book-Ratings.csv", sep=";", encoding="latin-1", on_bad_lines="skip", low_memory=False)

print("=== 1. SHAPES, DTYPES & MISSING VALUES ===")
for name, df in [("Users", users_eda), ("Books", books_eda), ("Ratings", ratings_eda)]:
    print(f"\n--- {name} (Shape: {df.shape}) ---")
    print(pd.DataFrame({"dtype": df.dtypes, "missing": df.isnull().sum()}))

print("\n=== 2. DISTRIBUTION OF RATINGS ===")
rating_counts = ratings_eda['Book-Rating'].value_counts().sort_index()
print(rating_counts)
zero_pct = (ratings_eda['Book-Rating'] == 0).mean() * 100
print(f"Percentage of 0 (implicit) ratings: {zero_pct:.2f}%")

print("\n=== 3. USER AGES & PUBLICATION YEARS ===")
print("User Age summary:\n", users_eda['Age'].describe())
print(f"Ages < 5: {(users_eda['Age'] < 5).sum()}, Ages > 100: {(users_eda['Age'] > 100).sum()}")

pub_years = pd.to_numeric(books_eda['Year-Of-Publication'], errors='coerce')
print("\nPublication Year summary:\n", pub_years.describe())
print(f"Year == 0: {(pub_years == 0).sum()}, Year > 2026: {(pub_years > 2026).sum()}")

print("\n=== 4. RATINGS PER USER AND PER BOOK ===")
explicit_ratings = ratings_eda[ratings_eda['Book-Rating'] > 0]
print("Explicit ratings per user quantiles:\n", explicit_ratings.groupby('User-ID').size().quantile([0.5, 0.75, 0.9, 0.95]))
print("Explicit ratings per book quantiles:\n", explicit_ratings.groupby('ISBN').size().quantile([0.5, 0.75, 0.9, 0.95]))

print("\n=== 5. MATRIX SPARSITY ===")
n_users = ratings_eda['User-ID'].nunique()
n_books = ratings_eda['ISBN'].nunique()
n_ratings = len(ratings_eda)
density = (n_ratings / (n_users * n_books)) * 100
print(f"Unique Users: {n_users:,}, Unique Books: {n_books:,}, Total Ratings: {n_ratings:,}")
print(f"Matrix Density (filled %): {density:.6f}% | Sparsity: {100 - density:.6f}%")

=== 1. SHAPES, DTYPES & MISSING VALUES ===

--- Users (Shape: (278858, 3)) ---
            dtype  missing
User-ID     int64        0
Location   object        0
Age       float64   110762

--- Books (Shape: (271360, 8)) ---
                      dtype  missing
ISBN                 object        0
Book-Title           object        0
Book-Author          object        2
Year-Of-Publication  object        0
Publisher            object        2
Image-URL-S          object        0
Image-URL-M          object        0
Image-URL-L          object        3

--- Ratings (Shape: (1149780, 3)) ---
              dtype  missing
User-ID       int64        0
ISBN         object        0
Book-Rating   int64        0

=== 2. DISTRIBUTION OF RATINGS ===
Book-Rating
0     716109
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610
Name: count, dtype: int64
Percentage of 0 (implicit) ratings: 62.28%

=== 3. USER AGES & PUBLICATI

###  Exploratory Data Analysis (EDA) Summary

* **Shape, data types, and missing values:** `Users` (~278k rows) has ~110k missing `Age` values, `Books` (~271k rows) has `Year-Of-Publication` stored as object/string with a few missing authors/publishers, and `Ratings` (~1.14M rows) has no nulls. In Silver, we will cast types properly and clean/impute missing fields.
* **Distribution of ratings:** Over 62% (~716k) of ratings are `0` (implicit interactions), while explicit ratings span `1–10` (skewed toward 7–10). In Silver, we will drop all `0` ratings to train exclusively on explicit preferences.
* **Distribution of user ages and publication years:** `Age` contains impossible values (`0` and values `> 100`, up to 244), and `Year-Of-Publication` contains `0` and future years (`> 2026`). In Silver, we will replace out-of-range ages (`< 5` or `> 100`) with the median age and set invalid publication years (`0` or `> 2026`) to null.
* **Number of ratings per user and per book:** The distribution is heavily long-tailed, with over 75% of users and books having fewer than 3 explicit ratings. In Silver, we will filter to users and books with at least 10 explicit ratings so cosine similarity has enough overlapping items to be meaningful.
* **Matrix sparsity:** Less than `0.004%` of possible user-book pairs contain a rating (over `99.996%` sparse). In Gold, we will represent the interaction matrix using `scipy.sparse.csr_matrix` to prevent memory overflow.

In [24]:
import pandas as pd
import numpy as np

# Read from Bronze
users = pd.read_parquet("bronze/users.parquet")
books = pd.read_parquet("bronze/books.parquet")
ratings = pd.read_parquet("bronze/ratings.parquet")

# Replace string 'nan'/'None' artifacts from Bronze serialization with true np.nan
for df in [users, books, ratings]:
    df.replace({'nan': np.nan, 'None': np.nan, '': np.nan}, inplace=True)

# 1. Remove duplicate rows
users = users.drop_duplicates(subset=['User-ID']).copy()
books = books.drop_duplicates(subset=['ISBN']).copy()
ratings = ratings.drop_duplicates(subset=['User-ID', 'ISBN']).copy()

# 2. Remove ratings of 0 and keep 1 to 10
ratings['User-ID'] = pd.to_numeric(ratings['User-ID'], errors='coerce')
ratings['Book-Rating'] = pd.to_numeric(ratings['Book-Rating'], errors='coerce')
ratings = ratings[(ratings['Book-Rating'] >= 1) & (ratings['Book-Rating'] <= 10)].dropna(subset=['User-ID', 'ISBN', 'Book-Rating'])
ratings['User-ID'] = ratings['User-ID'].astype(int)
ratings['Book-Rating'] = ratings['Book-Rating'].astype(int)

# 3. Fix user ages: < 5 or > 100 -> null, then fill with median
users['User-ID'] = pd.to_numeric(users['User-ID'], errors='coerce').astype(int)
users['Age'] = pd.to_numeric(users['Age'], errors='coerce')
users.loc[(users['Age'] < 5) | (users['Age'] > 100), 'Age'] = np.nan
median_age = users['Age'].median()
users['Age'] = users['Age'].fillna(median_age)

# 4. Split Location into city, state, country
loc_splits = users['Location'].fillna('').str.split(',', expand=True)
users['city'] = loc_splits[0].str.strip().str.lower().replace('', np.nan)
users['state'] = loc_splits[1].str.strip().str.lower().replace('', np.nan) if loc_splits.shape[1] > 1 else np.nan
users['country'] = loc_splits[2].str.strip().str.lower().replace('', np.nan) if loc_splits.shape[1] > 2 else np.nan
users = users.drop(columns=['Location'])

# 5. Fix publication years: 0 or future years (> 2026) -> null
books['Year-Of-Publication'] = pd.to_numeric(books['Year-Of-Publication'], errors='coerce')
books.loc[(books['Year-Of-Publication'] == 0) | (books['Year-Of-Publication'] > 2026), 'Year-Of-Publication'] = np.nan

# 6. Filter users and books with at least 10 explicit ratings
user_counts = ratings['User-ID'].value_counts()
book_counts = ratings['ISBN'].value_counts()
valid_users = user_counts[user_counts >= 10].index
valid_books = book_counts[book_counts >= 10].index

ratings_filtered = ratings[ratings['User-ID'].isin(valid_users) & ratings['ISBN'].isin(valid_books)].copy()
users_filtered = users[users['User-ID'].isin(valid_users)].copy()
books_filtered = books[books['ISBN'].isin(valid_books)].copy()

# Write cleaned Silver data
users_filtered.to_parquet("silver/users.parquet", index=False)
books_filtered.to_parquet("silver/books.parquet", index=False)
ratings_filtered.to_parquet("silver/ratings.parquet", index=False)

In [25]:
import os
import pickle
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

# 1. Load Silver data
ratings_df = pd.read_parquet("silver/ratings.parquet")
books_df = pd.read_parquet("silver/books.parquet")

# Deduplicate to guarantee a single rating per unique (User-ID, ISBN) pair
ratings_df = ratings_df.drop_duplicates(subset=['User-ID', 'ISBN'])

# 2. Create index mapping to maintain consistent alignment with sparse matrices
unique_users = sorted(ratings_df['User-ID'].unique())
unique_isbns = sorted(ratings_df['ISBN'].unique())

user_to_idx = {uid: idx for idx, uid in enumerate(unique_users)}
isbn_to_idx = {isbn: idx for idx, isbn in enumerate(unique_isbns)}
idx_to_isbn = {idx: isbn for isbn, idx in isbn_to_idx.items()}

# Convert indices to build sparse matrix
user_indices = ratings_df['User-ID'].map(user_to_idx).values
book_indices = ratings_df['ISBN'].map(isbn_to_idx).values
ratings_values = ratings_df['Book-Rating'].values

# 3. Create CSR Matrix (Books as rows, Users as columns for Item-Item CF)
# Shape: (n_books, n_users)
book_user_matrix = csr_matrix(
    (ratings_values, (book_indices, user_indices)),
    shape=(len(unique_isbns), len(unique_users))
)

# Save the sparse matrix and mappings into the Gold Layer directory
os.makedirs("gold", exist_ok=True)
with open("gold/book_user_matrix.pkl", "wb") as f:
    pickle.dump(book_user_matrix, f)

# 4. Fit the Collaborative Filtering model (Nearest Neighbors using Cosine Similarity)
model_knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=6)
model_knn.fit(book_user_matrix)

# Create a title lookup for easy demonstration
title_to_isbn = dict(zip(books_df['Book-Title'], books_df['ISBN']))
isbn_to_title = dict(zip(books_df['ISBN'], books_df['Book-Title']))

def recommend_books(book_title, n_recommendations=5):
    if book_title not in title_to_isbn:
        print(f"Book '{book_title}' not found in the silver dataset indices.")
        return

    isbn = title_to_isbn[book_title]
    if isbn not in isbn_to_idx:
        print(f"ISBN '{isbn}' for '{book_title}' is not present in our sparse matrix indices.")
        return

    book_idx = isbn_to_idx[isbn]
    distances, indices = model_knn.kneighbors(
        book_user_matrix[book_idx].reshape(1, -1),
        n_neighbors=n_recommendations + 1
    )

    print(f"Recommendations for '{book_title}':\n")
    for i in range(1, len(distances.flatten())):
        rec_isbn = idx_to_isbn[indices.flatten()[i]]
        rec_title = isbn_to_title.get(rec_isbn, "Unknown Title")
        print(f"{i}: {rec_title} (Distance: {distances.flatten()[i]:.4f})")

# Test the recommender system with a well-known title in the dataset
recommend_books("The Kitchen God's Wife")

Recommendations for 'The Kitchen God's Wife':

1: Woman of Independence (Distance: 0.7825)
2: The Joy Luck Club (Distance: 0.8001)
3: The Adventures of Tom Sawyer (Adventures of Tom Sawyer) (Distance: 0.8232)
4: The Hundred Secret Senses (Distance: 0.8360)
5: Second Wind (Distance: 0.8378)


### Threshold Justification

* **Chosen Threshold:** `MIN_RATINGS = 10` (both users and books must have at least 10 explicit ratings).
* **Justification from EDA:** In EDA, over 75% of users and books had fewer than 3 explicit ratings. Keeping users or books with only 1–2 ratings introduces severe noise into cosine similarity calculations (two books could appear 100% similar based on a single shared user) and keeps the interaction matrix over 99.996% sparse. Filtering to a minimum of 10 ratings retains dense, reliable reading patterns while preserving popular books and active readers.

In [26]:
import scipy.sparse as sp
import json

# Load clean data from Silver
users_s = pd.read_parquet("silver/users.parquet")
books_s = pd.read_parquet("silver/books.parquet")
ratings_s = pd.read_parquet("silver/ratings.parquet")

# 1. Create contiguous integer indices (0, 1, 2...) for Users (rows) and Books (columns)
unique_users = sorted(ratings_s['User-ID'].unique())
unique_books = sorted(ratings_s['ISBN'].unique())

user_to_idx = {uid: idx for idx, uid in enumerate(unique_users)}
book_to_idx = {isbn: idx for idx, isbn in enumerate(unique_books)}

row_ind = ratings_s['User-ID'].map(user_to_idx).values
col_ind = ratings_s['ISBN'].map(book_to_idx).values
data = ratings_s['Book-Rating'].astype(np.float32).values

# Build and save the Sparse Matrix (Rows = Users, Columns = Books)
user_book_matrix = sp.csr_matrix((data, (row_ind, col_ind)), shape=(len(unique_users), len(unique_books)))
sp.save_npz("gold/user_book_matrix.npz", user_book_matrix)

# Save index mappings so we can translate row/column numbers back to IDs later
with open("gold/mappings.json", "w") as f:
    json.dump({
        "users": [int(u) for u in unique_users],
        "books": unique_books
    }, f)

# 2. Build and save the Book Lookup Table (book_idx, ISBN, Title, Author)
book_lookup = pd.DataFrame({"book_idx": range(len(unique_books)), "ISBN": unique_books})
book_lookup = book_lookup.merge(books_s[['ISBN', 'Book-Title', 'Book-Author']], on='ISBN', how='left')
book_lookup.to_parquet("gold/book_lookup.parquet", index=False)

# 3. Build and save User-level and Book-level Aggregates
user_agg = ratings_s.groupby('User-ID').agg(
    user_avg_rating=('Book-Rating', 'mean'),
    user_rating_count=('Book-Rating', 'count')
).reset_index()

book_agg = ratings_s.groupby('ISBN').agg(
    book_avg_rating=('Book-Rating', 'mean'),
    book_rating_count=('Book-Rating', 'count')
).reset_index()

user_agg.to_parquet("gold/user_aggregates.parquet", index=False)
book_agg.to_parquet("gold/book_aggregates.parquet", index=False)

print("Gold Layer built successfully!")
print(f"Sparse Matrix Shape (Users x Books): {user_book_matrix.shape}")
print(f"Total ratings stored inside matrix: {user_book_matrix.nnz:,}")

Gold Layer built successfully!
Sparse Matrix Shape (Users x Books): (6837, 5640)
Total ratings stored inside matrix: 90,556


In [27]:
from sklearn.neighbors import NearestNeighbors

# 1. Load Gold artifacts and transpose matrix so Rows = Books, Columns = Users
user_book_matrix = sp.load_npz("gold/user_book_matrix.npz")
book_user_matrix = user_book_matrix.T.tocsr()  # Shape: (n_books, n_users)

book_lookup = pd.read_parquet("gold/book_lookup.parquet")
with open("gold/mappings.json", "r") as f:
    mappings = json.load(f)

idx_to_user = mappings["users"]
user_to_idx = {uid: idx for idx, uid in enumerate(idx_to_user)}
idx_to_book = mappings["books"]
book_to_idx = {isbn: idx for idx, isbn in enumerate(idx_to_book)}

# 2. Fit Cosine Similarity NearestNeighbors on the sparse book-user matrix
model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(book_user_matrix)

# 3. Function 1: Given a book title, return the 10 most similar books
def get_similar_books(book_title, n=10):
    # Case-insensitive exact match first, then partial match
    matches = book_lookup[book_lookup['Book-Title'].str.lower() == book_title.strip().lower()]
    if matches.empty:
        matches = book_lookup[book_lookup['Book-Title'].str.lower().str.contains(book_title.strip().lower(), na=False)]
    if matches.empty:
        return f"Book '{book_title}' not found in Gold catalog."

    target_idx = matches.iloc[0]['book_idx']
    matched_title = matches.iloc[0]['Book-Title']

    # Query n + 1 because index 0 is the book itself (distance = 0)
    distances, indices = model_knn.kneighbors(book_user_matrix[target_idx], n_neighbors=n + 1)

    results = []
    for dist, idx in zip(distances.flatten()[1:], indices.flatten()[1:]):
        row = book_lookup.iloc[idx]
        results.append({
            "Book-Title": row['Book-Title'],
            "Book-Author": row['Book-Author'],
            "Cosine_Similarity": round(1 - dist, 4)
        })
    print(f"Top {n} books similar to '{matched_title}':")
    return pd.DataFrame(results)

# 4. Function 2: Recommend top n unread books for a given user_id
def recommend(user_id, n=10, high_rating_threshold=7):
    if user_id not in user_to_idx:
        return f"User {user_id} not found in Gold dataset."

    u_idx = user_to_idx[user_id]
    user_ratings_row = user_book_matrix[u_idx].toarray().flatten()

    # Identify all books already read by the user
    read_book_indices = set(np.where(user_ratings_row > 0)[0])

    # Identify books the user rated highly (>= 7; fallback to user's above-average books if none)
    high_rated_indices = np.where(user_ratings_row >= high_rating_threshold)[0]
    if len(high_rated_indices) == 0:
        user_mean = user_ratings_row[list(read_book_indices)].mean()
        high_rated_indices = np.where(user_ratings_row >= user_mean)[0]

    # Accumulate similarity scores weighted by the user's rating
    candidate_scores = {}
    for b_idx in high_rated_indices:
        user_rating = user_ratings_row[b_idx]
        distances, indices = model_knn.kneighbors(book_user_matrix[b_idx], n_neighbors=21)

        for dist, neighbor_idx in zip(distances.flatten()[1:], indices.flatten()[1:]):
            if neighbor_idx in read_book_indices:
                continue  # Exclude books already read
            sim = 1.0 - dist
            if sim > 0:
                candidate_scores[neighbor_idx] = candidate_scores.get(neighbor_idx, 0.0) + (sim * user_rating)

    # Rank candidates by weighted similarity score descending
    ranked_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)[:n]

    recs = []
    for b_idx, score in ranked_candidates:
        row = book_lookup.iloc[b_idx]
        recs.append({
            "Recommended_Title": row['Book-Title'],
            "Author": row['Book-Author'],
            "Weighted_Score": round(score, 4)
        })
    return pd.DataFrame(recs)

print("Recommender model trained and ready!")

Recommender model trained and ready!


In [28]:
# Part A: Test item-to-item similarity on a well-known book
display(get_similar_books("The Da Vinci Code", n=5))

# Part B: Pick 3 users from the Gold dataset and compare their favorites vs. recommendations
sample_users = idx_to_user[:3]

for uid in sample_users:
    print("=" * 80)
    print(f"USER ID: {uid}")

    # Look up all books this user has already rated
    u_idx = user_to_idx[uid]
    u_row = user_book_matrix[u_idx].toarray().flatten()
    read_indices = np.where(u_row > 0)[0]

    liked_books = []
    for b_idx in read_indices:
        liked_books.append({
            "Liked_Book_Title": book_lookup.iloc[b_idx]['Book-Title'],
            "Author": book_lookup.iloc[b_idx]['Book-Author'],
            "User_Rating": int(u_row[b_idx])
        })
    liked_df = pd.DataFrame(liked_books).sort_values(by="User_Rating", ascending=False).head(5)

    print("\n--- Top 5 Books Already Rated by This User ---")
    display(liked_df)

    print("\n--- Top 10 New Books Recommended by Our Model ---")
    display(recommend(uid, n=10))

Top 5 books similar to 'The Da Vinci Code':


,Book-Title,Book-Author,Cosine_Similarity
0,Angels &amp; Demons,Dan Brown,0.2050
1,The Lovely Bones: A Novel,Alice Sebold,0.1376
2,Middlesex: A Novel,Jeffrey Eugenides,0.1346
3,Street Dreams,Faye Kellerman,0.1289
4,The Face,DEAN KOONTZ,0.1251


USER ID: 242

--- Top 5 Books Already Rated by This User ---


,Liked_Book_Title,Author,User_Rating
0,The Martian Chronicles,RAY BRADBURY,10
1,Veronika Deschliesst Zu Sterben / Vernika Deci...,Paolo Coelho,9



--- Top 10 New Books Recommended by Our Model ---


,Recommended_Title,Author,Weighted_Score
0,"Eine Frau, Eine Wohnung, Ein Roman",Wilhelm Genazino,4.6789
1,Free,Paul Vincent,4.6789
2,Jade Peony,Wayson Choy,4.6789
3,Due di due (Bestsellers),Andrea De Carlo,4.6789
4,None,None,4.0121
5,Bachelor brothers' bed &amp; breakfast,Bill Richardson,3.7223
6,Flight,Jan Burke,3.6536
7,God-Shaped Hole,Tiffanie DeBartolo,3.6206
8,"Schnee, der auf Zedern fÃ?Â¤llt.",David Guterson,3.4970
9,Das Zweite GedÃ?Â¤chtnis.,Ken Follett,3.4778


... (4 outputs left out)

USER ID: 254

--- Top 5 Books Already Rated by This User ---


,Liked_Book_Title,Author,User_Rating
5,American Gods,Neil Gaiman,10
0,Smoke and Mirrors: Short Fictions and Illusions,Neil Gaiman,9
6,American Gods: A Novel,Neil Gaiman,9
3,The Secret Life of Bees,Sue Monk Kidd,9
7,The Bonesetter's Daughter,Amy Tan,9



--- Top 10 New Books Recommended by Our Model ---


,Recommended_Title,Author,Weighted_Score
0,Harry Potter and the Order of the Phoenix (Boo...,J. K. Rowling,17.524401
1,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,11.509700
2,Harry Potter and the Sorcerer's Stone (Harry P...,J. K. Rowling,11.369200
3,"The Return of the King (The Lord of The Rings,...",J. R. R. Tolkien,11.220500
4,"The Two Towers (The Lord of the Rings, Part 2)",J. R. R. Tolkien,7.328500
5,Free,Paul Vincent,6.788600
6,Jade Peony,Wayson Choy,6.788600
7,Due di due (Bestsellers),Andrea De Carlo,6.788600
8,"Eine Frau, Eine Wohnung, Ein Roman",Wilhelm Genazino,6.788600
9,"The Amber Spyglass (His Dark Materials, Book 3)",Philip Pullman,5.832700
